# 02 — Normalize, validate, and prepare PxWeb data (v2: activity-style workflow)

**Purpose.** Walk through the lecture activity's workflow on the PxWeb data:

```
Understand  ->  Initial inspection / EDA  ->  Identify quality issues
            ->  Clean / transform  ->  EDA again  ->  Feature engineering
            ->  Prepare for the next stage
```

Each per-table section below follows this structure

- **Inputs:** `data/raw/<table_id>__data__slice_*.json`,
  `data/raw/<table_id>__meta.json`, `data/manifests/_dimension_catalog.json`,
  `configs/data.yaml`, `configs/data_quality.yaml`.
- **Outputs (per table):** `data/processed/<table_id>__normalized.csv`,
  `data/processed/<table_id>__quality.json`.
- **Aggregate outputs:** `data/processed/data_quality_summary.csv`,
  `data/processed/cross_table_coverage.csv`,
  `data/manifests/forecast_unit_catalog.json`.
- **Documentation:** `data/processed/outlier_policy.md`.

The 6 data-quality categories from the activity are:

1. Missing values
2. Incorrect or unexpected data types
3. Duplicate observations
4. Inconsistent categories
5. Potential outliers (report only — see `outlier_policy.md`)
6. Potentially irrelevant / redundant variables


## Setup: environment and config


In [ ]:
# mount Google Drive and set the working directory to the project path
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/JobAI")  # the project path
os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"

In [ ]:
import os, json, glob, csv, hashlib, datetime as dt
from pathlib import Path
from collections import defaultdict, Counter

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "data.yaml").is_file():
            return cand
    return p

REPO = _find_repo()
RAW  = REPO / "data" / "raw"
PRO  = REPO / "data" / "processed"
MAN  = REPO / "data" / "manifests"
for d in (PRO,):
    d.mkdir(parents=True, exist_ok=True)
NOW_UTC = dt.datetime.now(dt.timezone.utc)
TS = NOW_UTC.strftime("%Y%m%dT%H%M%SZ")
print("repo:", REPO)
print("ts  :", TS)


In [ ]:
import yaml
CFG = yaml.safe_load(open(REPO / "configs" / "data.yaml"))
TABLES_CFG = CFG["tables"]
INGEST_LIST = CFG["tables_to_ingest"]
DQ_CFG = yaml.safe_load(open(REPO / "configs" / "data_quality.yaml"))
DQ_TABLES = DQ_CFG["tables"]
OUTLIER_POLICY = DQ_CFG["outlier_policy"]
CAT = json.loads((MAN / "_dimension_catalog.json").read_text())
print("tables to process:", INGEST_LIST)
print("catalog tables   :", list(CAT))
print("outlier policy   :", OUTLIER_POLICY["rule"], "p<", OUTLIER_POLICY["threshold_percentile"])


## 1. Clean / transform

The activity's cleaning step. We do not remove outliers (see
`outlier_policy.md`). We do not impute missing values. The only
transformation we apply is the **KEHA monthly -> quarterly** aggregation
end-of-quarter rule, configured in `configs/data.yaml`.

This section produces the per-table `__normalized.csv` files. After this
runs, the dtype check in Section 3 will switch from "not yet produced" to
real values.


In [ ]:
def parse_jsonstat2(stat, table_id):
    ids = stat["id"]
    sizes = stat["size"]
    dims = stat["dimension"]
    raw_values = stat.get("value") or []
    n = 1
    for s in sizes:
        n *= s
    if len(raw_values) != n:
        raw_values = list(raw_values) + [None] * (n - len(raw_values))
    dim_categories = {}
    for d in ids:
        cat = dims[d]["category"]
        idx = cat["index"]
        labels = cat.get("label", {})
        codes = [None] * len(idx)
        for code, i in idx.items():
            codes[i] = code
        texts = [labels.get(c, c) for c in codes]
        dim_categories[d] = {"codes": codes, "texts": texts}
    strides = []
    acc = 1
    for s in reversed(sizes):
        strides.append(acc)
        acc *= s
    strides = list(reversed(strides))
    rows = []
    for flat, v in enumerate(raw_values):
        rem = flat
        coord = {}
        for d, s, st in zip(ids, sizes, strides):
            i = rem // st
            rem = rem % st
            coord[d] = {"code": dim_categories[d]["codes"][i], "text": dim_categories[d]["texts"][i]}
        if v is None or isinstance(v, (int, float)):
            num = v if isinstance(v, (int, float)) else None
            s = None
        else:
            num = None
            s = str(v)
        rows.append({
            "table_id": table_id,
            **{d: coord[d]["code"] for d in ids},
            **{f"{d}_text": coord[d]["text"] for d in ids},
            "value": num,
            "value_str": s,
            "source": stat.get("source", ""),
            "updated": stat.get("updated"),
        })
    return rows

print("parser ready")


In [ ]:
QUARTER_END_MONTHS = {"01": "Q1", "04": "Q2", "07": "Q3", "10": "Q4"}

def month_to_quarter(m):
    if not isinstance(m, str) or "M" not in m:
        return None
    yyyy, mm = m.split("M")
    if mm not in QUARTER_END_MONTHS:
        return None
    return f"{yyyy}{QUARTER_END_MONTHS[mm]}"

def aggregate_monthly_to_quarterly(rows, time_dim="timeperiod_m"):
    by_key = {}
    for r in rows:
        m = r.get(time_dim)
        q = month_to_quarter(m)
        if q is None:
            continue
        if r["value"] is None:
            continue
        other = tuple(sorted((k, v) for k, v in r.items() if k not in (time_dim, f"{time_dim}_text", "value", "value_str", "updated")))
        key = other + (("q", q),)
        if key not in by_key:
            base = dict(r)
            base["timeperiod_q"] = q
            base["timeperiod_m"] = None
            base["timeperiod_m_text"] = None
            base["agg_source_months"] = []
            by_key[key] = base
        by_key[key]["agg_source_months"].append(m)
    return list(by_key.values())

print("aggregator ready")


In [ ]:
table_slices = defaultdict(list)
for slice_path in sorted(RAW.glob("*__data__slice_*.json")):
    name = slice_path.name.replace("__data__slice_", "::").replace(".json", "")
    table_id = name.split("::")[0]
    table_slices[table_id].append(slice_path)

per_table_rows = {}
for tid, paths in table_slices.items():
    rows = []
    for p in paths:
        stat = json.loads(p.read_text())
        rows.extend(parse_jsonstat2(stat, tid))
    per_table_rows[tid] = rows
    print(f"  {tid:8s}  parsed {len(rows):>9,d} rows from {len(paths)} slices")

all_normalized = {}
for key in INGEST_LIST:
    tid = TABLES_CFG[key]["path"].split("/")[-1].replace(".px", "")
    rows = per_table_rows.get(tid, [])
    cfg_monthly = TABLES_CFG[key].get("monthly_to_quarterly")
    dims = list(CAT[tid])
    is_monthly = "timeperiod_m" in dims
    if is_monthly and cfg_monthly and cfg_monthly.get("rule") == "end_of_quarter":
        out_rows = aggregate_monthly_to_quarterly(rows, "timeperiod_m")
        time_col = "timeperiod_q"
        agg_note = "KEHA monthly aggregated to end-of-quarter (M03/M06/M09/M12)"
    else:
        out_rows = rows
        time_col = "timeperiod_q" if "timeperiod_q" in dims else ("timeperiod_m" if "timeperiod_m" in dims else None)
        agg_note = "no aggregation"
    all_normalized[tid] = {"rows": out_rows, "time_col": time_col, "agg_note": agg_note, "key": key}
    print(f"  {tid:8s}  {len(out_rows):>9,d} rows  time_col={time_col}  {agg_note}")

for tid, info in all_normalized.items():
    rows = info["rows"]
    if not rows:
        continue
    sample = rows[0]
    dim_cols = [k for k in sample if not k.endswith("_text") and k not in ("table_id","value","value_str","source","updated","agg_source_months")]
    text_cols = [f"{k}_text" for k in dim_cols if f"{k}_text" in sample]
    fieldnames = ["table_id"] + dim_cols + text_cols + ["value","value_str","source","updated"]
    if "agg_source_months" in sample:
        fieldnames.append("agg_source_months")
    csv_path = PRO / f"{tid}__normalized.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            out = dict(r)
            if "agg_source_months" in out:
                out["agg_source_months"] = ",".join(out["agg_source_months"])
            w.writerow(out)
    info["csv_path"] = csv_path
    info["n_dim_cols"] = len(dim_cols)
    print(f"  {tid:8s}  wrote {csv_path.name}  cols={len(fieldnames)}  rows={len(rows):,}")


## 2. Understanding the data

Before cleaning anything, we look at the catalogue of tables and the dimension
catalog. This is the activity's "Understanding the Dataset" step.


In [ ]:
print(f"{'table':12s} {'slices':>7s} {'dim_codes':<70s}  time_col")
for key in INGEST_LIST:
    tid = TABLES_CFG[key]["path"].split("/")[-1].replace(".px", "")
    dims = list(CAT[tid])
    n_slices = sum(1 for _ in (RAW / f"{tid}__data__slice_0.json").parent.glob(f"{tid}__data__slice_*.json"))
    is_monthly = "timeperiod_m" in dims
    time_col = "timeperiod_m" if is_monthly else "timeperiod_q"
    print(f"{tid:12s} {n_slices:>7d} {','.join(dims):<70s}  {time_col}")


## 3. Initial inspection and EDA

We follow the activity's three perspectives:

- **Univariate** — one variable at a time
- **Bivariate** — two variables together
- **Multivariate** — three or more

In our case the "variables" are the value series per (table, dimension tuple)
over time, so univariate EDA looks at the distribution of one series, and
bivariate EDA compares two regions or two measure codes.


### 3.1 Univariate: value distribution per table

We compute summary statistics and the count of distinct series per table.
This is the activity's `.describe()` step.


In [ ]:
import pandas as pd

try:
    from IPython.display import display as _display
except ImportError:
    def __display(x):
        if hasattr(x, 'to_string'):
            print(x.to_string())
        else:
            print(x)

summary_rows = []
for key in INGEST_LIST:
    tid = TABLES_CFG[key]["path"].split("/")[-1].replace(".px", "")
    csv_path = PRO / f"{tid}__normalized.csv"
    if not csv_path.exists():
        continue
    df = pd.read_csv(csv_path)
    v = pd.to_numeric(df["value"], errors="coerce").dropna()
    summary_rows.append({
        "table_id": tid,
        "n_rows": len(df),
        "n_unique_series": df.drop(columns=["timeperiod_q","timeperiod_m","value","value_str","source","updated","table_id","agg_source_months"], errors="ignore").drop_duplicates().shape[0] if len(df) else 0,
        "value_min": int(v.min()),
        "value_median": float(v.median()),
        "value_max": int(v.max()),
        "value_mean": float(v.mean()),
    })
summary_df = pd.DataFrame(summary_rows)
_display(summary_df)



### 3.2 Univariate: seasonality of total vacancies (11l1)

A histogram of `atp_lkm` (total vacancies) by quarter reveals the seasonal
pattern. Q2 and Q3 are typically lower (summer), Q4 and Q1 are typically
higher. 


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

df11l1 = pd.read_csv(PRO / "11l1__normalized.csv")
df11l1_total = df11l1[df11l1["contentscode"] == "atp_lkm"]
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df11l1_total, x="value", kde=True, bins=15, ax=ax)
ax.set_title("Distribution of Total Vacancies (11l1, atp_lkm)")
ax.set_xlabel("Vacancies per quarter")
ax.set_ylabel("Number of quarters")
plt.show()




### 3.3 Bivariate: total vacancies over time, colored by content code

The activity's bivariate step: how does one variable relate to another? We
plot total vacancies per quarter and add the hard-to-fill series to see how
the two move together.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=df11l1[df11l1["contentscode"].isin(["atp_lkm", "atp_vaik", "atp_eihoit"])],
    x="timeperiod_q", y="value", hue="contentscode", ax=ax,
)
ax.set_title("National vacancies per quarter (11l1)")
ax.set_xlabel("Quarter")
ax.set_ylabel("Vacancies")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



### 3.4 Multivariate: regional totals side by side

The activity's multivariate step: 11n1 gives us a panel across regions. We
plot total vacancies for each major region over time.


In [ ]:
import matplotlib
matplotlib.use("Agg")
df11n1 = pd.read_csv(PRO / "11n1__normalized.csv")
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=df11n1,
    x="timeperiod_q", y="value", hue="alue_16_20180101_text", ax=ax,
)
ax.set_title("Total vacancies by major region per quarter (11n1)")
ax.set_xlabel("Quarter")
ax.set_ylabel("Vacancies")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()



## 4. Identify data-quality problems

This is the activity's data-quality summary, but per-table. For each table
we walk through the 6 categories from the activity:

1. Missing values
2. Incorrect or unexpected data types
3. Duplicate observations
4. Inconsistent categories
5. Potential outliers (report only)
6. Potentially irrelevant / redundant variables

This section is **read-only** with respect to the data. It produces the
`data/processed/data_quality_summary.csv` artifact and per-table
`categories` keys in `__quality.json`.


In [ ]:
def discover_slices():
    out = defaultdict(list)
    for p in sorted(RAW.glob("*__data__slice_*.json")):
        name = p.name.replace("__data__slice_", "::").replace(".json", "")
        table_id = name.split("::")[0]
        out[table_id].append(p)
    return out

def parse_value_array(stat):
    sizes = stat["size"]
    raw = stat.get("value") or []
    n = 1
    for s in sizes:
        n *= s
    if len(raw) != n:
        raw = list(raw) + [None] * (n - len(raw))
    return raw, n

def category_observations(table_id):
    obs = []
    slices = discover_slices().get(table_id, [])
    if not slices:
        return [("missing_values", "no raw slice files found")]

    payloads = []
    null_in_value = 0
    str_in_value = 0
    total_cells = 0
    for p in slices:
        stat = json.loads(p.read_text())
        payloads.append(stat)
        vals, n = parse_value_array(stat)
        total_cells += n
        for v in vals:
            if v is None:
                null_in_value += 1
            elif not isinstance(v, (int, float)):
                str_in_value += 1

    if null_in_value == 0 and str_in_value == 0:
        obs.append(("missing_values", f"none: {total_cells:,} cells fully populated across {len(slices)} slice(s)"))
    else:
        obs.append(("missing_values", f"{null_in_value:,} null and {str_in_value:,} suppressed (string-marker) cells out of {total_cells:,}"))

    bad_labels = 0
    total_categories = 0
    for stat in payloads:
        for d, dim in (stat.get("dimension") or {}).items():
            cat = dim.get("category") or {}
            idx = cat.get("index") or {}
            labels = cat.get("label") or {}
            total_categories += len(idx)
            for code in idx.keys():
                if code in labels and labels[code] is None:
                    bad_labels += 1
    if bad_labels == 0:
        obs.append(("inconsistent_categories", f"all {total_categories:,} category labels present across {len(slices)} slice(s)"))
    else:
        obs.append(("inconsistent_categories", f"{bad_labels} category labels missing for {total_categories:,} total codes"))

    seen_keys = set()
    duplicates = 0
    n_rows = 0
    for stat in payloads:
        ids = stat["id"]
        sizes = stat["size"]
        dims = stat["dimension"]
        vals, n = parse_value_array(stat)
        if len(vals) != n:
            continue
        n_rows += n
        dim_cat = {}
        for d in ids:
            cat = dims[d]["category"]
            idx = cat["index"]
            codes = [None] * len(idx)
            for code, i in idx.items():
                codes[i] = code
            dim_cat[d] = codes
        strides = []
        acc = 1
        for s in reversed(sizes):
            strides.append(acc)
            acc *= s
        strides = list(reversed(strides))
        for flat, v in enumerate(vals):
            if v is None:
                continue
            rem = flat
            coord = []
            for d, s, st in zip(ids, sizes, strides):
                i = rem // st
                rem = rem % st
                coord.append((d, dim_cat[d][i]))
            key = tuple(coord)
            if key in seen_keys:
                duplicates += 1
            seen_keys.add(key)
    if duplicates == 0:
        obs.append(("duplicate_observations", f"none: {n_rows:,} non-null rows, {len(seen_keys):,} unique (dim-tuple) keys"))
    else:
        obs.append(("duplicate_observations", f"{duplicates} duplicate rows out of {n_rows:,}"))

    csv_path = PRO / f"{table_id}__normalized.csv"
    expected = DQ_TABLES.get(table_id, {}).get("expected_dtypes", {})
    if not expected:
        obs.append(("incorrect_unexpected_dtypes", "no expected_dtypes declared in configs/data_quality.yaml"))
    elif not csv_path.exists():
        obs.append(("incorrect_unexpected_dtypes", f"normalized CSV not yet produced (run Section 4 first)"))
    else:
        df = pd.read_csv(csv_path)
        mismatches = []
        for col, want in expected.items():
            if col not in df.columns:
                mismatches.append(f"{col}: missing column")
                continue
            got = str(df[col].dtype)
            if want == "Int64" and got in ("Int64", "int64", "float64"):
                continue
            if want == "object" and got in ("object", "str", "int64"):
                continue
            if got != want:
                mismatches.append(f"{col}: expected {want}, got {got}")
        if not mismatches:
            obs.append(("incorrect_unexpected_dtypes", f"all {len(expected)} expected dtypes match"))
        else:
            obs.append(("incorrect_unexpected_dtypes", "; ".join(mismatches)))

    threshold_pct = OUTLIER_POLICY["threshold_percentile"]
    if not csv_path.exists():
        obs.append(("potential_outliers", f"normalized CSV not yet produced; threshold p{threshold_pct} (report only)"))
    else:
        df = pd.read_csv(csv_path)
        if "value" not in df.columns:
            obs.append(("potential_outliers", "no `value` column in normalized CSV"))
        else:
            v = pd.to_numeric(df["value"], errors="coerce").dropna()
            if len(v) == 0:
                obs.append(("potential_outliers", "no numeric values to check"))
            else:
                thr = float(np.percentile(v, threshold_pct))
                n_above = int((v > thr).sum())
                pct_above = n_above / len(v) * 100
                obs.append(("potential_outliers", f"{n_above} values ({pct_above:.2f}%) above p{threshold_pct}={thr:,.0f} (REPORT ONLY - no action)"))

    redundant = DQ_TABLES.get(table_id, {}).get("redundant_variables", [])
    if not redundant:
        obs.append(("irrelevant_redundant_variables", "none declared in configs/data_quality.yaml"))
    else:
        obs.append(("irrelevant_redundant_variables", f"declared: {', '.join(redundant)}"))

    return obs

import numpy as np
print("category_observations() ready")


In [ ]:
per_table_obs = {}
for key in INGEST_LIST:
    tid = TABLES_CFG[key]["path"].split("/")[-1].replace(".px", "")
    per_table_obs[tid] = category_observations(tid)

print("=" * 88)
print("DATA-QUALITY SUMMARY (6 categories from the activity, per table)")
print("=" * 88)
for tid, obs in per_table_obs.items():
    print(f"\n{tid}  ({DQ_TABLES.get(tid, {}).get('title', '')})")
    for cat, observation in obs:
        print(f"  {cat:35s}  {observation}")

dq_summary_path = PRO / "data_quality_summary.csv"
with open(dq_summary_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["table_id", "category", "observation"])
    w.writeheader()
    for tid, obs in per_table_obs.items():
        for cat, observation in obs:
            w.writerow({"table_id": tid, "category": cat, "observation": observation})
print(f"\ndata-quality summary: {dq_summary_path}  ({sum(len(o) for o in per_table_obs.values())} rows)")


## 5. EDA again: validate the cleaning

The activity's "EDA Again" step. We re-run the per-table summary and the
dtype check to confirm the cleaning did what we expected, and we re-run the
data-quality summary to fill in the previously-empty cells.


In [ ]:
# Re-run the data-quality summary now that the normalized CSVs exist.
per_table_obs = {}
for key in INGEST_LIST:
    tid = TABLES_CFG[key]["path"].split("/")[-1].replace(".px", "")
    per_table_obs[tid] = category_observations(tid)

for tid, obs in per_table_obs.items():
    print(f"\n{tid}")
    for cat, observation in obs:
        print(f"  {cat:35s}  {observation}")


In [ ]:
# Per-table quality report (base §5.3 + 6-category extension).
def base_quality(tid, info):
    rows = info["rows"]
    rep = {"table_id": tid, "n_rows": len(rows)}
    sample = rows[0] if rows else {}
    dim_cols = [k for k in sample if not k.endswith("_text") and k not in ("table_id","value","value_str","source","updated","agg_source_months")]
    rep["dimensions"] = dim_cols
    rep["n_null_values"] = sum(1 for r in rows if r["value"] is None)
    rep["n_str_values"] = sum(1 for r in rows if r["value_str"] is not None)
    rep["negative_values"] = sum(1 for r in rows if isinstance(r["value"], (int,float)) and r["value"] < 0)
    time_col = info["time_col"]
    key_cols = dim_cols
    seen = set()
    dupes = 0
    for r in rows:
        k = tuple(r.get(c) for c in key_cols)
        if k in seen:
            dupes += 1
        seen.add(k)
    rep["n_duplicate_key_combos"] = dupes
    rep["n_unique_key_combos"] = len(seen)
    times = sorted({r.get(time_col) for r in rows if r.get(time_col)})
    rep["time_first"] = times[0] if times else None
    rep["time_last"]  = times[-1] if times else None
    rep["n_time_periods"] = len(times)
    rep["source"] = (sample.get("source") if sample else None)
    rep["updated"] = (sample.get("updated") if sample else None)
    return rep

quality = {}
for tid, info in all_normalized.items():
    rep = base_quality(tid, info)
    rep["categories"] = {cat: obs for cat, obs in per_table_obs.get(tid, [])}
    quality[tid] = rep
    qpath = PRO / f"{tid}__quality.json"
    qpath.write_text(json.dumps(rep, indent=2, ensure_ascii=False))
    print(f"  {tid:8s}  nulls={rep['n_null_values']:>7,d}  neg={rep['negative_values']:>5,d}  dupes={rep['n_duplicate_key_combos']:>5,d}  periods={rep['n_time_periods']:>4d}  {rep['time_first']}..{rep['time_last']}")


In [ ]:
fc = {}
for tid, info in all_normalized.items():
    sample = info["rows"][0] if info["rows"] else {}
    dim_cols = [k for k in sample if not k.endswith("_text") and k not in ("table_id","value","value_str","source","updated","agg_source_months")]
    time_col = info["time_col"]
    series_cols = [c for c in dim_cols if c != time_col and c not in ("timeperiod_m",)]
    fc[tid] = {
        "table_key": info["key"],
        "title": TABLES_CFG[info["key"]]["title"],
        "series_id_columns": series_cols,
        "time_column": time_col,
        "n_series": len({tuple(r.get(c) for c in series_cols) for r in info["rows"]}),
        "n_rows": len(info["rows"]),
    }
fc_path = MAN / "forecast_unit_catalog.json"
fc_path.write_text(json.dumps(fc, indent=2, ensure_ascii=False))
for tid, v in fc.items():
    print(f"  {tid:8s}  series_cols={v['series_id_columns']}  n_series={v['n_series']:>6,d}  n_rows={v['n_rows']:>7,d}")
print(f"forecast unit catalog: {fc_path}")


In [ ]:
# Cross-table coverage report.
cov_rows = []
for tid, info in all_normalized.items():
    rows = info["rows"]
    time_col = info["time_col"]
    sample = rows[0] if rows else {}
    dim_cols = [k for k in sample if not k.endswith("_text") and k not in ("table_id","value","value_str","source","updated","agg_source_months")]
    by_key = defaultdict(set)
    for r in rows:
        if r.get(time_col) is None:
            continue
        key = tuple((c, r.get(c)) for c in dim_cols if c != time_col)
        by_key[key].add(r[time_col])
    for k, times in by_key.items():
        sorted_times = sorted(times)
        cov_rows.append({
            "table_id": tid,
            "dimension_signature": "|".join(f"{c}={v}" for c, v in k),
            "n_time_periods": len(times),
            "time_first": sorted_times[0],
            "time_last": sorted_times[-1],
        })
cov_path = PRO / "cross_table_coverage.csv"
with open(cov_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["table_id","dimension_signature","n_time_periods","time_first","time_last"])
    w.writeheader()
    for r in cov_rows:
        w.writerow(r)
print(f"cross-table coverage: {len(cov_rows):>6,d} (table, dim-tuple) rows -> {cov_path.name}")


In [ ]:
total_rows = sum(v["n_rows"] for v in fc.values())
total_series = sum(v["n_series"] for v in fc.values())
print("FINAL DATASET SIZE")
print(f"  tables         : {len(fc)}")
print(f"  total rows     : {total_rows:>10,d}")
print(f"  total series   : {total_series:>10,d}")


In [ ]:
import re
import json as _json
try:
    from IPython.display import display as _display
except ImportError:
    def __display(x):
        if hasattr(x, 'to_string'):
            print(x.to_string())
        else:
            print(x)

rows = []
for tid, info in all_normalized.items():
    q = _json.loads(open(PRO / f"{tid}__quality.json").read())
    raw_slices = list((RAW).glob(f"{tid}__data__slice_*.json"))
    raw_cells = 0
    for p in raw_slices:
        st = _json.loads(p.read_text())
        n = 1
        for s in st["size"]:
            n *= s
        raw_cells += n
    mv = q["categories"]["missing_values"]
    m = re.search(r"([\d,]+)\s+null", mv)
    raw_nulls = int(m.group(1).replace(",", "")) if m else 0
    rows.append({
        "table_id": tid,
        "raw_cells": raw_cells,
        "normalized_rows": q["n_rows"],
        "n_unique_series": fc[tid]["n_series"],
        "n_time_periods": q["n_time_periods"],
        "time_first": q["time_first"],
        "time_last": q["time_last"],
        "raw_nulls": raw_nulls,
        "normalized_nulls": q["n_null_values"],
        "duplicates": q["n_duplicate_key_combos"],
    })
before_after = pd.DataFrame(rows)
_display(before_after)



In [ ]:
# Markdown-style "Before vs After" table mirroring the activity template.
print("| Table | Raw cells | Normalized rows | Series | Periods | Range |")
print("|---|---:|---:|---:|---:|---|")
for r in rows:
    print(f"| {r['table_id']} | {r['raw_cells']:,} | {r['normalized_rows']:,} | {r['n_unique_series']:,} | {r['n_time_periods']} | {r['time_first']} .. {r['time_last']} |")

print()
print("Cleaning actions applied:")
print("  - KEHA monthly -> quarterly (end-of-quarter rule)")
print("  - dtypes validated against configs/data_quality.yaml (tolerated pandas read_csv promotion)")
print("  - outlier counts reported, NO removal/capping (see outlier_policy.md)")
print()
print("Outputs:")
for f in sorted(PRO.glob("*")):
    if f.is_file():
        print(f"  {f.relative_to(REPO)}")
print()
print("Aggregate outputs:")
for f in sorted(MAN.glob("*")):
    if f.is_file():
        print(f"  {f.relative_to(REPO)}")
